# Music Annotation Project V3.2 Incremental Dataset Builder

**Runtime:** CPU.

This replacement Notebook 01 adds three controls needed for the expanded human-in-the-middle workflow:

1. **Persistent HITL train/validation allocation.** Existing configured validation pages remain validation pages. HITL pages receive a permanent 80/20 assignment stored in `pipeline_state/hitl_split_assignments.json`. Existing HITL pages are assigned retrospectively on the first run, so approximately one in five moves to validation.
2. **HITL-specific training-mask expansion.** Ordinary corrected masks continue to use the standard expansion settings from `project_config.json`. HITL masks use a milder configurable profile because the model draft and Gradio brush are already broad.
3. **Incremental rebuilding.** A page is rebuilt only when its aligned image, corrected mask, split assignment or relevant settings change.

The split assignment file is authoritative once created. Do not delete it casually, or historical HITL pages may receive different assignments.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

In [ ]:
import cv2,json,hashlib,re
import numpy as np
from pathlib import Path
from datetime import datetime,timezone

CONFIG_PATH=Path('/content/drive/MyDrive/MusicAnnotationProject/project_config.json')
if not CONFIG_PATH.exists():raise RuntimeError(f'Missing {CONFIG_PATH}')
CONFIG=json.loads(CONFIG_PATH.read_text())
ROOT=Path(CONFIG['project_root'])

ALIGNED_DIR=ROOT/'aligned';CORRECTED_MASK_DIR=ROOT/'masks'/'corrected';TRAINING_MASK_DIR=ROOT/'masks'/'training';DATASET_DIR=ROOT/'dataset'
TRAIN_IMAGES=DATASET_DIR/'images'/'train';VAL_IMAGES=DATASET_DIR/'images'/'val';TRAIN_LABELS=DATASET_DIR/'labels'/'train';VAL_LABELS=DATASET_DIR/'labels'/'val';TRAIN_MASK_TILES=DATASET_DIR/'mask_tiles'/'train';VAL_MASK_TILES=DATASET_DIR/'mask_tiles'/'val'
CORRECTED_REVIEW=ROOT/'review'/'corrected_masks';TRAINING_REVIEW=ROOT/'review'/'training_masks';TILE_REVIEW=ROOT/'review'/'tiles';HITL_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit';STATE_DIR=ROOT/'pipeline_state';STATE_PATH=STATE_DIR/'notebook1_dataset_state.json';HITL_SPLIT_PATH=STATE_DIR/'hitl_split_assignments.json';YAML=ROOT/'dataset_v3.yaml';MANIFEST=ROOT/'dataset_manifest_v3.json'

FORCE_REBUILD_ALL=False
REMOVE_OUTPUTS_FOR_DELETED_MASKS=True
MASK_NONZERO_THRESHOLD=0
MIN_CLEAN_COMPONENT_AREA=30
HITL_VALIDATION_RATE=0.20

# Standard masks retain the project configuration.
STANDARD_EXPANSION={
 'dilation_size':int(CONFIG['training_mask']['dilation_size']),
 'dilation_iterations':int(CONFIG['training_mask']['dilation_iterations']),
 'closing_size':int(CONFIG['training_mask']['closing_size']),
}
# HITL masks are already broad. Start here and review the training-mask overlays.
HITL_EXPANSION={
 'dilation_size':3,
 'dilation_iterations':1,
 'closing_size':5,
}

TILE_SIZE=int(CONFIG['tile_size']);TILE_STRIDE=int(CONFIG['tile_stride']);LABEL=CONFIG['labels'];FIXED_VALIDATION=set(CONFIG.get('validation_pages',[]));SUPPORTED={'.jpg','.jpeg','.png','.tif','.tiff'}
for folder in [ALIGNED_DIR,CORRECTED_MASK_DIR,TRAINING_MASK_DIR,TRAIN_IMAGES,VAL_IMAGES,TRAIN_LABELS,VAL_LABELS,TRAIN_MASK_TILES,VAL_MASK_TILES,CORRECTED_REVIEW,TRAINING_REVIEW,TILE_REVIEW,HITL_AUDIT_DIR,STATE_DIR]:folder.mkdir(parents=True,exist_ok=True)
print('Standard expansion:',STANDARD_EXPANSION)
print('HITL expansion:',HITL_EXPANSION)
print('HITL validation target:',HITL_VALIDATION_RATE)

## Identify HITL pages and create permanent split assignments

HITL pages are recognised from `human_in_the_middle/audit/Page###.json`. Assignments already stored are never changed. New or previously unassigned HITL pages are processed in page-number order, with every fifth HITL page assigned to validation.

This provides a stable 80/20 split. It does not solve source-set leakage by itself. If several pages are consecutive pages from the same score, later versions should add a `source_group` field to the audit and split by group.

In [ ]:
def image_files(folder):return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)
def find_image(folder,page):
 hits=[p for p in image_files(folder) if p.stem.lower()==page.lower()]
 return hits[0] if len(hits)==1 else None
def md5(path):
 h=hashlib.md5()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def object_hash(value):return hashlib.md5(json.dumps(value,sort_keys=True).encode()).hexdigest()
def read_json(path,default):
 try:return json.loads(Path(path).read_text()) if Path(path).exists() else default
 except Exception:return default
def write_json_atomic(path,value):
 temp=Path(str(path)+'.tmp');temp.write_text(json.dumps(value,indent=2));temp.replace(path)
def page_sort_key(page):
 m=re.search(r'(\d+)$',page);return int(m.group(1)) if m else 10**9

hitl_pages=sorted({p.stem for p in HITL_AUDIT_DIR.glob('Page*.json')},key=page_sort_key)
assignments=read_json(HITL_SPLIT_PATH,{'version':1,'assignments':{}})
split_map=assignments.get('assignments',{})
# Preserve existing assignments; allocate all new HITL pages in stable sequence.
assigned_hitl=sorted([p for p in hitl_pages if p in split_map],key=page_sort_key)
new_hitl=sorted([p for p in hitl_pages if p not in split_map],key=page_sort_key)
sequence_index=len(assigned_hitl)
for page in new_hitl:
 sequence_index+=1
 split_map[page]='val' if sequence_index%5==0 else 'train'
assignments={'version':2,'updated_utc':datetime.now(timezone.utc).isoformat(),'validation_rate':HITL_VALIDATION_RATE,'method':'persistent_every_fifth_hitl_page','assignments':split_map}
write_json_atomic(HITL_SPLIT_PATH,assignments)
print('HITL pages:',len(hitl_pages),'train:',sum(split_map.get(p)=='train' for p in hitl_pages),'val:',sum(split_map.get(p)=='val' for p in hitl_pages))
print('Split file:',HITL_SPLIT_PATH)

## Validate pages and calculate incremental fingerprints

In [ ]:
def normalise_corrected(mask):
 binary=np.where(mask>MASK_NONZERO_THRESHOLD,255,0).astype(np.uint8)
 n,labels,stats,_=cv2.connectedComponentsWithStats(binary,8);clean=np.zeros_like(binary)
 for i in range(1,n):
  if stats[i,cv2.CC_STAT_AREA]>=MIN_CLEAN_COMPONENT_AREA:clean[labels==i]=255
 return clean

def split_for(page):
 if page in split_map:return split_map[page]
 return 'val' if page in FIXED_VALIDATION else 'train'
def route_for(page):return 'hitl' if page in split_map else 'standard'
def expansion_for(page):return HITL_EXPANSION if route_for(page)=='hitl' else STANDARD_EXPANSION

def training_mask(mask,profile):
 result=mask.copy()
 if profile['dilation_size']>1 and profile['dilation_iterations']>0:
  kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(profile['dilation_size'],profile['dilation_size']))
  result=cv2.dilate(result,kernel,iterations=profile['dilation_iterations'])
 if profile['closing_size']>1:
  kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(profile['closing_size'],profile['closing_size']))
  result=cv2.morphologyEx(result,cv2.MORPH_CLOSE,kernel)
 return result

valid={};problems=[]
for mask_path in image_files(CORRECTED_MASK_DIR):
 page=mask_path.stem
 if not re.fullmatch(r'Page\d+',page,re.I):continue
 aligned_path=find_image(ALIGNED_DIR,page)
 if aligned_path is None:problems.append((page,'aligned image missing'));continue
 aligned=cv2.imread(str(aligned_path),0);raw=cv2.imread(str(mask_path),0)
 if aligned is None or raw is None or aligned.shape!=raw.shape:problems.append((page,'unreadable or shape mismatch'));continue
 mask=normalise_corrected(raw)  # Empty mask remains valid as a hard negative.
 split=split_for(page);route=route_for(page);profile=expansion_for(page)
 fingerprint={'aligned_md5':md5(aligned_path),'corrected_md5':md5(mask_path),'split':split,'route':route,'expansion':profile,'tile_size':TILE_SIZE,'tile_stride':TILE_STRIDE,'label_settings':LABEL,'minimum_clean_component_area':MIN_CLEAN_COMPONENT_AREA}
 valid[page]={'aligned_path':aligned_path,'corrected_path':mask_path,'mask':mask,'split':split,'route':route,'profile':profile,'fingerprint':fingerprint}
print('Valid pages:',len(valid),'Problems:',problems)
print('Train pages:',sum(v['split']=='train' for v in valid.values()),'Validation pages:',sum(v['split']=='val' for v in valid.values()))

## Rebuild changed pages only

In [ ]:
def positions(n):
 if n<=TILE_SIZE:return [0]
 p=list(range(0,n-TILE_SIZE+1,TILE_STRIDE))
 if p[-1]!=n-TILE_SIZE:p.append(n-TILE_SIZE)
 return p
def mask_to_lines(mask):
 contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE);lines=[];h,w=mask.shape
 for contour in contours:
  if cv2.contourArea(contour)<LABEL['minimum_contour_area']:continue
  poly=cv2.approxPolyDP(contour,max(1.0,LABEL['polygon_epsilon_fraction']*cv2.arcLength(contour,True)),True);values=[]
  for pt in poly:values.extend([pt[0][0]/w,pt[0][1]/h])
  if len(values)>=6:lines.append('0 '+' '.join(f'{v:.6f}' for v in values))
 return lines
def overlay(base,mask,alpha=.55):
 colour=cv2.cvtColor(base,cv2.COLOR_GRAY2BGR) if base.ndim==2 else base.copy();layer=colour.copy();layer[mask>0]=(0,0,255);return cv2.addWeighted(colour,1-alpha,layer,alpha,0)
def remove_page_outputs(page):
 for p in [TRAINING_MASK_DIR/f'{page}.png',CORRECTED_REVIEW/f'{page}.png',TRAINING_REVIEW/f'{page}.png']:
  if p.exists():p.unlink()
 for folder,pattern in [(TRAIN_IMAGES,f'{page}_y*_x*.jpg'),(VAL_IMAGES,f'{page}_y*_x*.jpg'),(TRAIN_MASK_TILES,f'{page}_y*_x*.png'),(VAL_MASK_TILES,f'{page}_y*_x*.png'),(TRAIN_LABELS,f'{page}_y*_x*.txt'),(VAL_LABELS,f'{page}_y*_x*.txt'),(TILE_REVIEW,f'*_{page}_y*_x*.jpg')]:
  for p in folder.glob(pattern):p.unlink()

state=read_json(STATE_PATH,{'pages':{}});old=state.get('pages',{});to_build=[];skipped=[]
def outputs_exist(page,split):
 idir=TRAIN_IMAGES if split=='train' else VAL_IMAGES;return (TRAINING_MASK_DIR/f'{page}.png').exists() and any(idir.glob(f'{page}_y*_x*.jpg'))
for page,info in valid.items():
 unchanged=(not FORCE_REBUILD_ALL and old.get(page,{}).get('fingerprint')==info['fingerprint'] and outputs_exist(page,info['split']))
 (skipped if unchanged else to_build).append(page)
print('Pages to rebuild:',to_build);print('Unchanged pages skipped:',len(skipped))
for page in to_build:remove_page_outputs(page)
if REMOVE_OUTPUTS_FOR_DELETED_MASKS:
 for page in sorted(set(old)-set(valid)):remove_page_outputs(page)

new_state={};records=[]
for page,info in valid.items():
 if page not in to_build:
  new_state[page]=old[page];continue
 mask=info['mask'];train_mask=training_mask(mask,info['profile']);cv2.imwrite(str(TRAINING_MASK_DIR/f'{page}.png'),train_mask)
 aligned_gray=cv2.imread(str(info['aligned_path']),0);aligned_colour=cv2.cvtColor(aligned_gray,cv2.COLOR_GRAY2BGR)
 cv2.imwrite(str(CORRECTED_REVIEW/f'{page}.png'),overlay(aligned_colour,mask,.65));cv2.imwrite(str(TRAINING_REVIEW/f'{page}.png'),overlay(aligned_colour,train_mask,.55))
 split=info['split'];idir=TRAIN_IMAGES if split=='train' else VAL_IMAGES;mdir=TRAIN_MASK_TILES if split=='train' else VAL_MASK_TILES;ldir=TRAIN_LABELS if split=='train' else VAL_LABELS;tile_count=positive=zero_polygon=0
 for y in positions(train_mask.shape[0]):
  for x in positions(train_mask.shape[1]):
   image_tile=aligned_colour[y:y+TILE_SIZE,x:x+TILE_SIZE];mask_tile=train_mask[y:y+TILE_SIZE,x:x+TILE_SIZE]
   if image_tile.shape[:2]!=(TILE_SIZE,TILE_SIZE):continue
   name=f'{page}_y{y:04d}_x{x:04d}';ip=idir/f'{name}.jpg';mp=mdir/f'{name}.png';lp=ldir/f'{name}.txt';cv2.imwrite(str(ip),image_tile,[cv2.IMWRITE_JPEG_QUALITY,95]);cv2.imwrite(str(mp),mask_tile);lines=mask_to_lines(mask_tile);lp.write_text('\n'.join(lines)+('\n' if lines else ''));tile_count+=1;is_positive=bool(np.any(mask_tile>0));positive+=int(is_positive);zero_polygon+=int(is_positive and not lines)
 new_state[page]={'fingerprint':info['fingerprint'],'processed_utc':datetime.now(timezone.utc).isoformat(),'split':split,'route':info['route'],'expansion':info['profile'],'tile_count':tile_count,'positive_tiles':positive,'positive_zero_polygon':zero_polygon}
 print('Rebuilt',page,split,info['route'],info['profile'],'tiles',tile_count)

## Validate the accumulated dataset and refresh manifests

In [ ]:
train_pages=sorted(p for p,v in valid.items() if v['split']=='train');val_pages=sorted(p for p,v in valid.items() if v['split']=='val');all_records=[]
for split,idir,mdir,ldir,pages in [('train',TRAIN_IMAGES,TRAIN_MASK_TILES,TRAIN_LABELS,train_pages),('val',VAL_IMAGES,VAL_MASK_TILES,VAL_LABELS,val_pages)]:
 images={p.stem:p for p in image_files(idir)};masks={p.stem:p for p in image_files(mdir)};labels={p.stem:p for p in ldir.glob('*.txt')}
 allowed=set(pages)
 for stem in sorted(set(images)|set(masks)|set(labels)):
  m=re.match(r'^(.*)_y\d+_x\d+$',stem);page=m.group(1) if m else None
  if page not in allowed:
   for coll in [images,masks,labels]:
    if stem in coll:coll[stem].unlink()
 images={p.stem:p for p in image_files(idir)};masks={p.stem:p for p in image_files(mdir)};labels={p.stem:p for p in ldir.glob('*.txt')}
 if set(images)!=set(masks) or set(images)!=set(labels):raise RuntimeError(f'{split} image/mask/label mismatch')
 for stem in sorted(images):
  mask=cv2.imread(str(masks[stem]),0);all_records.append({'tile_name':stem,'split':split,'positive':bool(np.any(mask>0)),'mask_pixels':int(np.sum(mask>0)),'polygon_count':sum(bool(line.strip()) for line in labels[stem].read_text().splitlines())})
YAML.write_text(f'path: {DATASET_DIR}\ntrain: images/train\nval: images/val\nnames:\n  0: annotation\n')
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'incremental':True,'training_pages':train_pages,'validation_pages':val_pages,'fixed_validation_pages':sorted(FIXED_VALIDATION),'hitl_split_file':str(HITL_SPLIT_PATH),'hitl_validation_rate':HITL_VALIDATION_RATE,'standard_expansion':STANDARD_EXPANSION,'hitl_expansion':HITL_EXPANSION,'pages':[{'page':p,'split':valid[p]['split'],'route':valid[p]['route'],'expansion':valid[p]['profile'],'aligned':str(valid[p]['aligned_path']),'corrected':str(valid[p]['corrected_path']),'training':str(TRAINING_MASK_DIR/f'{p}.png')} for p in train_pages+val_pages],'tiles':all_records}
MANIFEST.write_text(json.dumps(manifest,indent=2));write_json_atomic(STATE_PATH,{'version':3,'updated_utc':datetime.now(timezone.utc).isoformat(),'pages':new_state})
print('Training pages:',len(train_pages));print('Validation pages:',len(val_pages));print('HITL train:',sum(valid[p]['route']=='hitl' for p in train_pages),'HITL val:',sum(valid[p]['route']=='hitl' for p in val_pages));print('Tiles:',len(all_records));print('DATASET READY FOR GPU TRAINING')